In [4]:
!pip install transformers datasets torch evaluate rouge_score
!pip install -U datasets


In [6]:
import wandb
wandb.init(mode="disabled")
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, Trainer, TrainingArguments, pipeline
from datasets import load_dataset
import evaluate

# 1. Load the SQuAD dataset
dataset = load_dataset("squad")
small_train = dataset["train"].select(range(1000))
small_validation = dataset["validation"].select(range(200))

# 2. Load the tokenizer and model
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

def preprocess_function(examples):
    questions = [q.strip() for q in examples["question"]]
    contexts = [c.strip() for c in examples["context"]]

    # Tokenize inputs
    tokenized = tokenizer(
        questions,
        contexts,
        max_length=384,
        truncation="only_second",
        stride=128,
        return_offsets_mapping=True,
        padding="max_length",
        return_tensors="pt"
    )

    # Initialize arrays for start and end positions
    start_positions = []
    end_positions = []

    # Process each example
    for i in range(len(questions)):
        # Get the answer's start and end positions in the context
        answer_start = examples['answers'][i]['answer_start'][0]
        answer_end = answer_start + len(examples['answers'][i]['text'][0])

        # Get the offset mapping for this example
        offset = tokenized.offset_mapping[i]

        # Find the start token
        start_position = 0
        for idx, (start, end) in enumerate(offset):
            if start <= answer_start <= end:
                start_position = idx
                break

        # Find the end token
        end_position = 0
        for idx, (start, end) in enumerate(offset):
            if start <= answer_end <= end:
                end_position = idx
                break

        start_positions.append(start_position)
        end_positions.append(end_position)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions

    return tokenized

# Process datasets
tokenized_train = small_train.map(
    preprocess_function,
    batched=True,
    remove_columns=small_train.column_names,
    batch_size=8
)

tokenized_validation = small_validation.map(
    preprocess_function,
    batched=True,
    remove_columns=small_validation.column_names,
    batch_size=8
)

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_strategy="steps",
    logging_steps=50,
    logging_first_step=True
)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation
)

# Train the model
trainer.train()

# Save the model
model.save_pretrained("./qa_bert_model")
tokenizer.save_pretrained("./qa_bert_model")

# Test the model
qa_pipeline = pipeline("question-answering", model="./qa_bert_model", tokenizer="./qa_bert_model")

# Example
context = "Hugging Face Inc. is a company based in New York City. It is famous for creating the Transformers library."
question = "Where is Hugging Face based?"

prediction = qa_pipeline(question=question, context=context)
print(f"\nQuestion: {question}")
print(f"Answer: {prediction['answer']}")

Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Step,Training Loss
1,5.966000
50,4.962800
100,4.206200
150,3.694000
200,3.160000
250,2.975700


Device set to use cuda:0



Question: Where is Hugging Face based?
Answer: Hugging Face Inc


In [8]:
context = "Hugging Face is a company based in New York City. It is famous for creating the Transformers library."
question = "Where is Hugging Face based?"

prediction = qa_pipeline(question=question, context=context)
print(f"\nQuestion: {question}")
print(f"Answer: {prediction['answer']}")


Question: Where is Hugging Face based?
Answer: New York City. It is famous for creating the Transformers library
